# QProfiler Tutorial

This tutorial demonstrates how to use QProfiler to benchmark machine learning models and analyze data complexity.

## What is QProfiler?

QProfiler is an automated ML benchmarking tool that provides:

- **Model Performance Evaluation**: Tests both classical and quantum ML algorithms
- **Data Complexity Analysis**: Computes 15+ intrinsic dataset characteristics
- **Correlation Analysis**: Links model performance to data properties
- **Automated Workflows**: Handles data splitting, scaling, and evaluation

## Tutorial Overview

In this tutorial, you will:

1. Generate artificial datasets for testing
2. Configure QProfiler with a YAML file
3. Run QProfiler to benchmark multiple models
4. Visualize and analyze the results
5. Understand data complexity metrics

## 1. Setup and Imports

First, let's import the necessary modules.

In [ ]:
import sys
import os
import re

# Import QBioCode
import qbiocode as qbc

# For running QProfiler
import yaml

## 2. Generate Test Data

We'll create a simple artificial dataset to demonstrate QProfiler's capabilities.

**Dataset Parameters:**
- 100 samples
- 10 features (2 informative, 2 redundant)
- 2 classes with 2 clusters each
- 3 different class weight distributions

In [ ]:
type_of_data = 'classes'

N_SAMPLES = [100]
N_FEATURES = [10]
N_INFORMATIVE = [2]
N_REDUNDANT = [2]
N_CLASSES = [2]
N_CLUSTERS_PER_CLASS = [2]
WEIGHTS = [[0.3, 0.7], [0.4, 0.6], [0.5, 0.5]]

qbc.generate_data(
    type_of_data=type_of_data,
    save_path=os.path.join('data', 'ld_data'),
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_INFORMATIVE,
    n_redundant=N_REDUNDANT,
    n_classes=N_CLASSES,
    n_clusters_per_class=N_CLUSTERS_PER_CLASS,
    weights=WEIGHTS,
    random_state=42
)

print(f"Generated {len(WEIGHTS)} datasets in data/ld_data/")

## 3. Configure QProfiler

QProfiler uses a YAML configuration file to specify:

- **Data directories**: Where to find your datasets
- **Models to test**: Which ML algorithms to benchmark
- **Embeddings**: Dimensionality reduction methods to try
- **Output settings**: Where to save results

The configuration file `configs/config.yaml` contains all these settings.

### Key Configuration Options:

**Models:**
- Classical: Random Forest (RF), SVM (SVC), Logistic Regression (LR), Decision Tree (DT), Naive Bayes (NB), MLP
- Quantum: QSVC, PQK, VQC, QNN (requires quantum backend)

**Embeddings:**
- none, pca, lle, isomap, spectral, umap, nmf

**Data Complexity Metrics:**
- Intrinsic dimension, fractal dimension
- Statistical properties (variance, skewness, kurtosis)
- Separability measures (Fisher ratio, mutual information)

## 4. Run QProfiler

There are two ways to run QProfiler:

### Option 1: Command Line

```bash
qprofiler --config configs/config.yaml
```

### Option 2: Python API (shown below)

This approach loads the config and runs QProfiler programmatically.

In [ ]:
# Load configuration
config = yaml.safe_load(open('configs/config.yaml', 'r'))

# Import QProfiler
from apps.qprofiler import qprofiler as profiler

# Run QProfiler with the configuration
profiler.main(config)

print("QProfiler execution complete!")
print("Results saved to:")
print("  - ModelResults.csv")
print("  - RawDataEvaluation.csv")
print("  - compiled_results.csv")

## 5. Analyze Results

QProfiler generates several output files:

### Output Files:

1. **ModelResults.csv**: Performance metrics for each model
   - Accuracy, F1-score, AUC, precision, recall
   - Training and test times
   - Model-specific parameters

2. **RawDataEvaluation.csv**: Data complexity metrics
   - Intrinsic dimension, fractal dimension
   - Statistical properties
   - Separability measures

3. **compiled_results.csv**: Combined model and data metrics
   - Enables correlation analysis
   - Links performance to data characteristics

4. **Performance plots**: Visualizations in `performance_summary_and_spearman_correlation_plots/`

Let's load and examine the results.

In [ ]:
import pandas as pd

# Load model results
model_results = pd.read_csv('ModelResults.csv')
print("Model Performance Results:")
print(model_results[['model', 'accuracy', 'f1_score', 'auc']].head())

# Load data complexity metrics
data_eval = pd.read_csv('RawDataEvaluation.csv')
print("\nData Complexity Metrics:")
print(data_eval.columns.tolist())

## 6. Visualize Results

QProfiler automatically generates several visualizations:

### Generated Plots:

1. **Boxplots**: Compare model performance across datasets
   - Accuracy, F1-score, AUC distributions
   
2. **Correlation Heatmaps**: Show relationships between:
   - Data complexity metrics
   - Model performance
   
3. **Best Model Summary**: Identify top performers per dataset

Let's create some custom visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")

# Plot model comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Accuracy comparison
sns.boxplot(data=model_results, x='model', y='accuracy', ax=axes[0])
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# F1-score comparison
sns.boxplot(data=model_results, x='model', y='f1_score', ax=axes[1])
axes[1].set_title('Model F1-Score Comparison')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

# AUC comparison
sns.boxplot(data=model_results, x='model', y='auc', ax=axes[2])
axes[2].set_title('Model AUC Comparison')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 7. Understanding Data Complexity Metrics

QProfiler computes various metrics to characterize your data:

### Geometric Properties:
- **Intrinsic Dimension**: True dimensionality of data (often < number of features)
- **Fractal Dimension**: Measures self-similarity and complexity

### Statistical Properties:
- **Variance**: Data spread across features
- **Skewness**: Distribution asymmetry
- **Kurtosis**: Tail heaviness

### Separability Measures:
- **Fisher Discriminant Ratio**: Class separability (higher = more separable)
- **Mutual Information**: Feature-label dependence

These metrics help explain why certain models perform better on specific datasets.

In [ ]:
# Examine data complexity for our datasets
print("Data Complexity Summary:")
print(data_eval[['dataset', 'intrinsic_dim', 'fractal_dim', 'fisher_ratio']].head())

# Correlation between complexity and performance
compiled = pd.read_csv('compiled_results.csv')
correlation = compiled[['intrinsic_dim', 'fractal_dim', 'fisher_ratio', 
                        'accuracy', 'f1_score']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation: Data Complexity vs Model Performance')
plt.tight_layout()
plt.show()

## Summary

In this tutorial, you learned how to:

1. ✅ Generate artificial datasets for testing
2. ✅ Configure QProfiler with YAML files
3. ✅ Run QProfiler to benchmark multiple ML models
4. ✅ Analyze model performance results
5. ✅ Visualize and compare model performance
6. ✅ Understand data complexity metrics
7. ✅ Correlate data properties with model performance

## Next Steps

- **Try different datasets**: Use your own data or generate more complex artificial datasets
- **Experiment with embeddings**: Test different dimensionality reduction methods
- **Quantum models**: If you have access to quantum hardware, try QSVC, PQK, VQC
- **Grid search**: For quantum models, use `generate_qml_experiment_configs()` utility
- **Batch processing**: Run QProfiler on multiple datasets using bash loops or SLURM

## See Also

- [QProfiler Documentation](../../apps/profiler.rst) - Full configuration reference
- [Data Generation Tutorial](../Artificial_data_generation/example_data_generation.ipynb) - Create more datasets
- [API Documentation](https://ibm.github.io/QBioCode/api/qbiocode.html) - Complete API reference